In [1]:
print("hello")

hello


In [2]:
# ============================================================
# EXPERIMENT 2 — ECOBOTX LIGHTWEIGHT BACKBONE
# ============================================================

from pathlib import Path

CUSTOM_YAML = Path(
    r"G:\EcoBotX_Object_Detection\ecobotx_light.yaml"
)

yaml_text = """
# EcoBotX-Light
# Experiment 2
# Lightweight backbone for Raspberry Pi deployment

nc: 4

depth_multiple: 0.33
width_multiple: 0.25

backbone:

  # [from, repeats, module, args]

  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]

  # Lightweight replacement for C2f
  - [-1, 3, C3Ghost, [128, True]]

  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 6, C3Ghost, [256, True]]

  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 6, C3Ghost, [512, True]]

  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 3, C3Ghost, [1024, True]]

  - [-1, 1, SPPF, [1024, 5]]

head:

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 3, C3Ghost, [512]]

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 3, C3Ghost, [256]]

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 3, C3Ghost, [512]]

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 3, C3Ghost, [1024]]

  - [[15, 18, 21], 1, Detect, [nc]]
"""

CUSTOM_YAML.write_text(yaml_text)

print("=" * 70)
print("ECOBOTX-LIGHT YAML CREATED")
print("=" * 70)
print("Path:", CUSTOM_YAML)

ECOBOTX-LIGHT YAML CREATED
Path: G:\EcoBotX_Object_Detection\ecobotx_light.yaml


In [3]:
# ============================================================
# BUILD ECOBOTX-LIGHT
# ============================================================

from ultralytics import YOLO

custom_model = YOLO(str(CUSTOM_YAML))

print("=" * 70)
print("ECOBOTX-LIGHT MODEL CREATED")
print("=" * 70)

custom_model.info(verbose=True)

ECOBOTX-LIGHT MODEL CREATED
ecobotx_light summary: 226 layers, 1,999,344 parameters, 1,999,328 gradients, 5.7 GFLOPs


(226, 1999344, 1999328, 5.6904704)

In [5]:
# ============================================================
# COMPARE YOLOv8n VS ECOBOTX-LIGHT
# ============================================================

from ultralytics import YOLO

# Load the trained YOLOv8n baseline again
BASELINE_MODEL = r"G:\EcoBotX_YOLO_training\yolov8n_ecobotx\weights\best.pt"

baseline_model = YOLO(BASELINE_MODEL)

print("=" * 70)
print("YOLOv8n BASELINE")
print("=" * 70)

baseline_model.info(verbose=True)

print("\n" + "=" * 70)
print("EcoBotX-Light")
print("=" * 70)

custom_model.info(verbose=True)

YOLOv8n BASELINE
Model summary: 130 layers, 3,011,628 parameters, 0 gradients, 8.2 GFLOPs

EcoBotX-Light
ecobotx_light summary: 226 layers, 1,999,344 parameters, 1,999,328 gradients, 5.7 GFLOPs


(226, 1999344, 1999328, 5.6904704)

In [6]:
# ============================================================
# EXPERIMENT 2 — ECOBOTX-LIGHT TRAINING
# ============================================================

from pathlib import Path
from ultralytics import YOLO

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

CUSTOM_YAML = r"G:\EcoBotX_Object_Detection\ecobotx_light.yaml"

DATASET_YAML = r"G:\EcoBotX_YOLO_FINAL\dataset.yaml"

PROJECT_DIR = r"G:\EcoBotX_YOLO_training"

# ------------------------------------------------------------
# Load architecture
# ------------------------------------------------------------

eco_model = YOLO(CUSTOM_YAML)

print("=" * 70)
print("EXPERIMENT 2 — ECOBOTX-LIGHT")
print("=" * 70)

eco_model.info(verbose=True)

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

results_exp2 = eco_model.train(

    # Dataset
    data=DATASET_YAML,

    # Training
    epochs=100,
    imgsz=640,
    batch=8,

    # GPU
    device=0,

    # Data loading
    workers=4,

    # Training from scratch
    pretrained=False,

    # Optimizer
    optimizer="auto",

    # Early stopping
    patience=20,

    # Saving
    save=True,
    save_period=10,

    # Plots
    plots=True,

    # Output
    project=PROJECT_DIR,
    name="experiment2_ecobotx_light",

    verbose=True
)

print("\n" + "=" * 70)
print("EXPERIMENT 2 TRAINING COMPLETE")
print("=" * 70)

EXPERIMENT 2 — ECOBOTX-LIGHT
ecobotx_light summary: 226 layers, 1,999,344 parameters, 1,999,328 gradients, 5.7 GFLOPs
Ultralytics 8.4.126  Python-3.12.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=G:\EcoBotX_YOLO_FINAL\dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mas

KeyboardInterrupt: 